# 04 - Fetch Reference Authors

In this notebook, I turn the paper level reference lists into a citation graph.

For every referenced OpenAlex work, I fetch the cited work metadata and then
explode its authorships. The output has one row for each cited author on each
referenced work. This is the table needed for citation count outcomes and for
the citation data check plots.

The first full run is the slowest Step 2 notebook because it fetches many cited
works from OpenAlex. The notebook is resumable: each cited work is saved as a
separate JSON file before the next request is made.


## 1 - Setup

In [ ]:
import json
import os
import re
import socket
import time

from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path

import pandas as pd
import requests
import pyalex
from pyalex import Works
from tqdm.auto import tqdm


In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids
openalex_fetch_workers = int(openalex_config.get("fetch_workers", 8))
step_2_data_dir = project_folder / "step_2_data"
step_2_artifacts_dir = project_folder / "step_2_artifacts"
prepared_dir = step_2_data_dir / "prepared"
intermediate_dir = step_2_data_dir / "intermediate"
raw_dir = step_2_data_dir / "raw"
summary_tables_dir = step_2_artifacts_dir / "summary_tables"
dependency_tables_dir = step_2_artifacts_dir / "dependency_tables"
check_tables_dir = step_2_artifacts_dir / "check_tables"

all_papers_path = prepared_dir / "all_papers" / "all_papers_filtered.parquet"
work_cache_dir = raw_dir / "openalex_cache" / "works"
openalex_cache_dir = raw_dir / "openalex_cache"
reference_dir = intermediate_dir / "references"
pldi_intermediate_dir = intermediate_dir / "pldi_papers"

prepared_all_dir = prepared_dir / "all_papers"
prepared_pacmpl_dir = prepared_dir / "pacmpl_papers"

for path in [
    openalex_cache_dir,
    work_cache_dir,
    reference_dir,
    pldi_intermediate_dir,
    prepared_all_dir,
    prepared_pacmpl_dir,
    summary_tables_dir,
]:
    path.mkdir(parents=True, exist_ok=True)

REFRESH_DATA = overwrite_data
SAMPLE_LIMIT = openalex_sample_limit


def write_data_parquet(df, path):
    if path.exists() and not overwrite_data:
        print(f"kept existing data file: {path}")
        return False
    df.to_parquet(path, index=False)
    print(f"wrote data file: {path}")
    return True


def write_artifact_csv(df, path):
    if path.exists() and not overwrite_artifacts:
        print(f"kept existing artifact file: {path}")
        return False
    df.to_csv(path, index=False)
    print(f"wrote artifact file: {path}")
    return True


def load_env_file(path):
    if not path.exists():
        return False
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value
    return True


loaded_env_files = [
    str(path.relative_to(project_folder))
    for path in [project_folder / ".env", project_folder / "key.env"]
    if load_env_file(path)
]

HAS_OPENALEX_KEY = bool(os.environ.get("OPENALEX_API_KEY"))

if HAS_OPENALEX_KEY:
    pyalex.config.api_key = os.environ["OPENALEX_API_KEY"]
    FETCH_SLEEP = 0.05
else:
    FETCH_SLEEP = 0.20

pyalex.config.max_retries = 2
pyalex.config.retry_backoff_factor = 0.5
pyalex.config.retry_http_codes = [429, 500, 502, 503, 504]

MAX_RETRIES = 4
RETRY_WAIT = 20
CACHE_SNAPSHOT_DATE = os.environ.get("OPENALEX_SNAPSHOT_DATE") or date.today().isoformat()

socket.setdefaulttimeout(30)

print(project_folder)
print(f"Run mode: {run_mode}")
print(f"Loaded local env files: {loaded_env_files}")
print(f"OpenAlex API key configured: {HAS_OPENALEX_KEY}")
print(f"Fetch sleep: {FETCH_SLEEP} seconds")
print(f"Cache snapshot date: {CACHE_SNAPSHOT_DATE}")
print("OpenAlex reference-work fetch: singleton endpoint")
print(f"OpenAlex fetch workers: {openalex_fetch_workers}")
if openalex_sample_include_work_ids:
    print(f"Forced sample work IDs: {openalex_sample_include_work_ids}")



## 2 - Helper Functions

In [ ]:
def short_openalex_id(value):
    if not value:
        return None
    return str(value).rstrip("/").rsplit("/", 1)[-1]


def short_orcid(orcid):
    if not isinstance(orcid, str) or not orcid.strip():
        return None
    return orcid.rstrip("/").rsplit("/", 1)[-1]


DATE_FOLDER_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def cache_lookup_path(work_id):
    work_id = short_openalex_id(work_id)
    if work_id is None:
        return None
    key = Path("works") / f"{work_id}.json"

    dated_dirs = sorted(
        [
            path
            for path in openalex_cache_dir.iterdir()
            if path.is_dir() and DATE_FOLDER_RE.match(path.name)
        ],
        reverse=True,
    )
    for dated_dir in dated_dirs:
        path = dated_dir / key
        if path.exists():
            return path

    flat_path = work_cache_dir / f"{work_id}.json"
    if flat_path.exists():
        return flat_path

    return None


def cache_write_path(work_id):
    work_id = short_openalex_id(work_id)
    if work_id is None:
        return None
    return openalex_cache_dir / CACHE_SNAPSHOT_DATE / "works" / f"{work_id}.json"


def write_cached_work(work_id, data):
    path = cache_write_path(work_id)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, default=str))
    return path


def read_cached_work(work_id):
    path = cache_lookup_path(work_id)
    if path is None or not path.exists():
        return None
    try:
        data = json.loads(path.read_text())
    except json.JSONDecodeError:
        return "CORRUPT_CACHE"
    if not isinstance(data, dict):
        return None
    return data


def fetch_single_openalex_work(work_id):
    work_id = short_openalex_id(work_id)
    if not work_id:
        return None

    url = f"https://api.openalex.org/works/{work_id}"
    params = {}
    if os.environ.get("OPENALEX_API_KEY"):
        params["api_key"] = os.environ["OPENALEX_API_KEY"]

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 404:
                return None
            if response.status_code in {429, 500, 502, 503, 504}:
                if attempt == MAX_RETRIES:
                    return None
                time.sleep(RETRY_WAIT)
                continue
            response.raise_for_status()
            data = response.json()
            write_cached_work(work_id, data)
            if FETCH_SLEEP > 0:
                time.sleep(FETCH_SLEEP)
            return data
        except Exception:
            if attempt == MAX_RETRIES:
                return None
            time.sleep(RETRY_WAIT)

    return None


def extract_cited_authorships(work):
    rows = []
    for authorship in work.get("authorships") or []:
        author = authorship.get("author") or {}
        rows.append({
            "ref_author_name": author.get("display_name"),
            "ref_author_id": short_openalex_id(author.get("id")),
            "ref_orcid": short_orcid(author.get("orcid") or authorship.get("raw_orcid")),
            "author_position": authorship.get("author_position") or authorship.get("position"),
        })
    return rows


def citing_author_ids(authorships):
    out = set()
    if authorships is None:
        return out
    try:
        iterator = list(authorships)
    except TypeError:
        return out
    for authorship in iterator:
        if isinstance(authorship, dict):
            author_id = authorship.get("author_id")
            if author_id:
                out.add(author_id)
    return out


## 3 - Build Citing Paper to Cited Work Edges

In [ ]:
papers = pd.read_parquet(all_papers_path).copy()

papers["venue"] = papers["conference"].where(papers["conference"].eq("PLDI"), "PACMPL")

print(f"papers: {len(papers):,}")
print(f"conference-year cells: {papers[['conference', 'conference_year']].drop_duplicates().shape[0]:,}")
print(f"reference links in paper metadata: {papers['referenced_works'].apply(len).sum():,}")

display(
    papers
    .groupby(["conference", "conference_year"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "conference_year"])
)


In [ ]:
edge_rows = []

for paper in papers.itertuples(index=False):
    refs = getattr(paper, "referenced_works")
    if refs is None:
        continue
    for ref in refs:
        ref_id = short_openalex_id(ref)
        if not ref_id:
            continue
        edge_rows.append({
            "work_id": paper.work_id,
            "doi": paper.doi,
            "title": paper.title,
            "conference": paper.conference,
            "conference_year": paper.conference_year,
            "issue": paper.issue,
            "source": paper.source,
            "venue": paper.venue,
            "referenced_work_id": ref_id,
        })

reference_edges = pd.DataFrame(edge_rows).drop_duplicates()

reference_edges_path = reference_dir / "reference_edges.parquet"

write_data_parquet(reference_edges, reference_edges_path)

print(f"reference edges: {len(reference_edges):,}")
print(f"unique cited works: {reference_edges['referenced_work_id'].nunique():,}")
print(reference_edges_path)

display(reference_edges.head())


## 4 - Fetch Referenced Works from OpenAlex

In [ ]:
all_reference_ids = sorted(reference_edges["referenced_work_id"].dropna().unique())
all_reference_id_set = set(all_reference_ids)

forced_reference_ids = []
for value in openalex_sample_include_work_ids:
    forced_id = short_openalex_id(value)
    if forced_id in all_reference_id_set and forced_id not in forced_reference_ids:
        forced_reference_ids.append(forced_id)

missing_forced_reference_ids = [
    short_openalex_id(value)
    for value in openalex_sample_include_work_ids
    if short_openalex_id(value) not in all_reference_id_set
]

if SAMPLE_LIMIT is None:
    reference_ids = all_reference_ids
else:
    if len(forced_reference_ids) > SAMPLE_LIMIT:
        raise ValueError("openalex.sample_limit is smaller than the number of forced sample work IDs")
    base_reference_ids = [work_id for work_id in all_reference_ids if work_id not in set(forced_reference_ids)]
    reference_ids = base_reference_ids[:SAMPLE_LIMIT - len(forced_reference_ids)] + forced_reference_ids

print(f"unique cited works in data: {len(all_reference_ids):,}")
print(f"unique cited works in this run: {len(reference_ids):,}")
print(f"forced sample work IDs present: {forced_reference_ids}")
if missing_forced_reference_ids:
    print(f"forced sample work IDs not found in references: {missing_forced_reference_ids}")

reference_id_set = set(reference_ids)
edges_for_run = reference_edges[reference_edges["referenced_work_id"].isin(reference_id_set)].copy()
print(f"reference edges in this run: {len(edges_for_run):,}")


In [ ]:
def fetch_openalex_works(work_ids):
    works_by_id = {}
    status_by_id = {}
    ids_to_fetch = []

    for work_id in work_ids:
        work_id = short_openalex_id(work_id)
        existing_path = cache_lookup_path(work_id)

        if existing_path is not None and not REFRESH_DATA:
            cached = read_cached_work(work_id)
            if cached != "CORRUPT_CACHE":
                works_by_id[work_id] = cached
                status_by_id[work_id] = "cached" if isinstance(cached, dict) else "cached_not_found"
                continue

        ids_to_fetch.append(work_id)

    if ids_to_fetch and not allow_network:
        raise RuntimeError(
            "This run needs OpenAlex network access, but config inputs.allow_network is false. "
            "Use run_mode: full with inputs.allow_network: true, or provide the cached OpenAlex works."
        )

    if ids_to_fetch:
        print(f"fetching from OpenAlex with {openalex_fetch_workers} workers")

    with ThreadPoolExecutor(max_workers=openalex_fetch_workers) as executor:
        future_to_work_id = {
            executor.submit(fetch_single_openalex_work, work_id): work_id
            for work_id in ids_to_fetch
        }
        for future in tqdm(as_completed(future_to_work_id), total=len(future_to_work_id), desc="OpenAlex cited-work singleton"):
            work_id = future_to_work_id[future]
            try:
                data = future.result()
            except Exception:
                data = None
            if isinstance(data, dict):
                works_by_id[work_id] = data
                status_by_id[work_id] = "fetched_singleton"
            else:
                write_cached_work(work_id, None)
                works_by_id[work_id] = None
                status_by_id[work_id] = "not_found"

    return works_by_id, status_by_id


In [ ]:
fetch_rows = []

cached_before = sum(cache_lookup_path(work_id) is not None and not REFRESH_DATA for work_id in reference_ids)
to_fetch = len(reference_ids) - cached_before
print(f"cached before this run: {cached_before:,}")
print(f"to fetch this run: {to_fetch:,}")

works_by_id, status_by_id = fetch_openalex_works(reference_ids)

for work_id in reference_ids:
    work = works_by_id.get(work_id)
    status = status_by_id.get(work_id, "missing_status")
    fetch_rows.append({
        "referenced_work_id": work_id,
        "fetch_status": status,
        "has_work": isinstance(work, dict),
        "n_authorships": len((work or {}).get("authorships") or []) if isinstance(work, dict) else 0,
        "publication_year": (work or {}).get("publication_year") if isinstance(work, dict) else None,
    })

fetch_summary = pd.DataFrame(fetch_rows)

if SAMPLE_LIMIT is None:
    fetch_summary_path = summary_tables_dir / "reference_author_fetch_summary.csv"
else:
    fetch_summary_path = summary_tables_dir / f"reference_author_fetch_summary_sample_{SAMPLE_LIMIT}.csv"

write_artifact_csv(fetch_summary, fetch_summary_path)

display(fetch_summary["fetch_status"].value_counts().rename_axis("fetch_status").reset_index(name="n"))
print(fetch_summary_path)


## 5 - Explode Cited Authors

In [ ]:
citing_authors_by_work = {
    row.work_id: citing_author_ids(row.authorships)
    for row in papers.itertuples(index=False)
}

exploded_columns = [
    "work_id",
    "doi",
    "referenced_work_id",
    "conference",
    "conference_year",
    "issue",
    "source",
    "venue",
    "ref_author_name",
    "ref_author_id",
    "ref_orcid",
    "cited_publication_year",
    "author_position",
    "is_self_citation",
]

exploded_rows = []
missing_work_rows = 0
missing_author_rows = 0

for edge in tqdm(edges_for_run.itertuples(index=False), total=len(edges_for_run), desc="explode cited authors"):
    cited_work = read_cached_work(edge.referenced_work_id)
    if cited_work is None:
        missing_work_rows += 1
        continue

    cited_authors = extract_cited_authorships(cited_work)
    if not cited_authors:
        missing_author_rows += 1
        continue

    citing_ids = citing_authors_by_work.get(edge.work_id, set())
    cited_publication_year = cited_work.get("publication_year")

    for author in cited_authors:
        ref_author_id = author["ref_author_id"]
        exploded_rows.append({
            "work_id": edge.work_id,
            "doi": edge.doi,
            "referenced_work_id": edge.referenced_work_id,
            "conference": edge.conference,
            "conference_year": edge.conference_year,
            "issue": edge.issue,
            "source": edge.source,
            "venue": edge.venue,
            "ref_author_name": author["ref_author_name"],
            "ref_author_id": ref_author_id,
            "ref_orcid": author["ref_orcid"],
            "cited_publication_year": cited_publication_year,
            "author_position": author["author_position"],
            "is_self_citation": ref_author_id is not None and ref_author_id in citing_ids,
        })

all_ref_authors = pd.DataFrame(exploded_rows, columns=exploded_columns)

print(f"exploded rows: {len(all_ref_authors):,}")
print(f"reference edges with missing cited-work metadata: {missing_work_rows:,}")
print(f"reference edges with cited work but no authorships: {missing_author_rows:,}")

if len(all_ref_authors):
    print(f"unique citing papers: {all_ref_authors['work_id'].nunique():,}")
    print(f"unique cited works: {all_ref_authors['referenced_work_id'].nunique():,}")
    print(f"unique cited authors: {all_ref_authors['ref_author_id'].nunique():,}")
    print(f"self-citation rows: {int(all_ref_authors['is_self_citation'].sum()):,}")
    display(all_ref_authors.head())


## 6 - Coverage Checks

In [ ]:
if len(all_ref_authors):
    cited_work_author_counts = (
        all_ref_authors
        .groupby("referenced_work_id")
        .size()
        .rename("n_cited_author_rows")
        .reset_index()
    )
else:
    cited_work_author_counts = pd.DataFrame(columns=["referenced_work_id", "n_cited_author_rows"])

edge_check = edges_for_run.merge(cited_work_author_counts, on="referenced_work_id", how="left")
edge_check["n_cited_author_rows"] = edge_check["n_cited_author_rows"].fillna(0).astype(int)
edge_check["has_cited_author_metadata"] = edge_check["n_cited_author_rows"].gt(0)

coverage = (
    edge_check
    .groupby(["conference", "conference_year"], as_index=False)
    .agg(
        n_papers=("work_id", "nunique"),
        n_reference_edges=("referenced_work_id", "size"),
        n_unique_cited_works=("referenced_work_id", "nunique"),
        n_edges_with_author_metadata=("has_cited_author_metadata", "sum"),
    )
)

coverage["share_edges_with_author_metadata"] = (
    coverage["n_edges_with_author_metadata"] / coverage["n_reference_edges"]
).round(4)

if SAMPLE_LIMIT is None:
    coverage_path = summary_tables_dir / "author_coverage_by_year.csv"
else:
    coverage_path = summary_tables_dir / f"author_coverage_by_year_sample_{SAMPLE_LIMIT}.csv"

write_artifact_csv(coverage, coverage_path)

display(coverage.sort_values(["conference", "conference_year"]))
print(coverage_path)


## 7 - Write Outputs

In [ ]:
if SAMPLE_LIMIT is None:
    all_output_path = prepared_all_dir / "all_ref_authors_exploded.parquet"
    pacmpl_output_path = prepared_pacmpl_dir / "pacmpl_ref_authors_exploded.parquet"
    pldi_output_path = pldi_intermediate_dir / "pldi_ref_authors_exploded.parquet"
else:
    all_output_path = prepared_all_dir / f"all_ref_authors_exploded_sample_{SAMPLE_LIMIT}.parquet"
    pacmpl_output_path = prepared_pacmpl_dir / f"pacmpl_ref_authors_exploded_sample_{SAMPLE_LIMIT}.parquet"
    pldi_output_path = pldi_intermediate_dir / f"pldi_ref_authors_exploded_sample_{SAMPLE_LIMIT}.parquet"

base_output_columns = [
    "work_id",
    "doi",
    "referenced_work_id",
    "conference_year",
    "issue",
    "ref_author_name",
    "ref_author_id",
    "ref_orcid",
    "cited_publication_year",
    "author_position",
    "is_self_citation",
]

all_ref_authors_output = all_ref_authors[base_output_columns + ["venue"]].copy()
pacmpl_ref_authors = (
    all_ref_authors[all_ref_authors["venue"].eq("PACMPL")]
    [base_output_columns]
    .copy()
)
pldi_ref_authors = (
    all_ref_authors[all_ref_authors["conference"].eq("PLDI")]
    [base_output_columns]
    .copy()
)

write_data_parquet(all_ref_authors_output, all_output_path)

write_data_parquet(pacmpl_ref_authors, pacmpl_output_path)
write_data_parquet(pldi_ref_authors, pldi_output_path)

print(all_output_path)
print(pacmpl_output_path)
print(pldi_output_path)

print(f"all rows: {len(all_ref_authors_output):,}")
print(f"PACMPL rows: {len(pacmpl_ref_authors):,}")
print(f"PLDI rows: {len(pldi_ref_authors):,}")


## 8 - Output Files

In [ ]:
outputs = pd.DataFrame([
    {"path": str(reference_edges_path.relative_to(repo)), "description": "One row per citing paper and cited OpenAlex work"},
    {"path": str(all_output_path.relative_to(repo)), "description": "One row per cited author on each referenced work"},
    {"path": str(pacmpl_output_path.relative_to(repo)), "description": "PACMPL subset of the exploded cited-author table"},
    {"path": str(pldi_output_path.relative_to(repo)), "description": "PLDI subset of the exploded cited-author table"},
    {"path": str(fetch_summary_path.relative_to(repo)), "description": "Fetch status for each cited OpenAlex work"},
    {"path": str(coverage_path.relative_to(repo)), "description": "Conference-year coverage check"},
])

display(outputs)
